# 02 · TestGen — KG → seeds → evolve → difficulty
The generative centrepiece of the project.

In [ ]:

import sys, pathlib
ROOT = pathlib.Path().resolve().parent if pathlib.Path('notebooks').exists() else pathlib.Path().resolve()
sys.path.insert(0, str(ROOT))


In [ ]:
from src.ingest.loader import _Document
from src.ingest.chunker import chunk_documents
from src.testgen.llm import MockLLM  # swap for get_llm() when ANTHROPIC_API_KEY is set
from src.testgen.knowledge_graph import extract_knowledge_graph
from src.testgen.seed_generator import generate_seeds
from src.testgen.evol_instruct import evolve_batch
from src.testgen.difficulty_matrix import compute_difficulty_matrix, difficulty_breakdown_report
from src.testgen.groundtruth import verify_groundedness

llm = MockLLM()
docs = [_Document('Alpha uses Rust. Bob leads Alpha. Beta uses Go. Alice leads Beta.' * 20,
        {'source_file': f'd{i}.txt', 'page_number':0,'file_type':'txt'}) for i in range(5)]
chunks = chunk_documents(docs, chunk_size=256, overlap_ratio=0.1)


## Knowledge graph

In [ ]:
kg = extract_knowledge_graph(chunks, llm)
print('nodes:', len(kg.nodes), 'edges:', len(kg.edges))
for f in kg.get_facts()[:5]:
    print('FACT:', f['fact'])


## Seeds

In [ ]:
seeds = generate_seeds(kg, llm, num_seeds=20, chunks=chunks)
for s in seeds[:5]:
    print('-', s['question'])


## Evolve (4 types)

In [ ]:
evolved = evolve_batch(seeds, kg, llm)
from collections import Counter
print(Counter(q['question_type'] for q in evolved))
for q in evolved[:4]:
    print(q['question_type'], '|', q['question'][:140])


## Groundedness filter + difficulty matrix

In [ ]:
kept = []
for q in evolved:
    ok, conf = verify_groundedness(q['question'], q['ground_truth_answer'], q['ground_truth_context'], llm)
    if ok and conf >= 0.5:
        kept.append(q)
print('kept:', len(kept), 'of', len(evolved))
df = compute_difficulty_matrix(kept)
print(difficulty_breakdown_report(df))
df[['question','question_type','reasoning_depth','semantic_distance','difficulty']].head()
